<a href="https://colab.research.google.com/github/naza-campioni/bait_bycatch_analysis/blob/main/test_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
DRIVE_PATH = '/content/figures/'

import os
os.makedirs(DRIVE_PATH, exist_ok=True)

In [2]:
if not os.path.exists('/content/bait_bycatch_analysis'):
    !git clone https://github.com/naza-campioni/bait_bycatch_analysis.git \
        /content/bait_bycatch_analysis

Cloning into '/content/bait_bycatch_analysis'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 31 (delta 6), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 670.27 KiB | 5.88 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [3]:
import sys
sys.path.insert(0, '/content/bait_bycatch_analysis')

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import mannwhitneyu, chi2_contingency, fisher_exact
from statsmodels.stats.power import tt_ind_solve_power
from scipy.stats import shapiro

import pymc as pm
import arviz as az

import importlib

from functions.calc_contingency import calc_contingency
from functions.calc_manwu import calc_manwu
from functions.subset import subset
from functions.univariate_tests import run_tests, rank_biserial

## 1. Aim of the study

Blue sharks in this dataset were caught using two different baits:
mackerel ("Sgombro", n = 108) and an artificial, squid-shaped lure filled with frozen sardines ("Totano", n = 50). We investigate whether bait type at capture is associated with any measured biological or environmental variable. If an association exists, it could lead to implications for selective bycatch reduction.

### Hypotheses
One null hypothesis per variable tested. All tests are two-sided.

H₀₁: The distribution of depth is identical between bait groups  
H₀₂: The distribution of SST is identical between bait groups  
H₀₃: The distribution of chlorophyll is identical between bait groups  
H₀₄: The distribution of NPPV is identical between bait groups  
H₀₅: The distribution of dissolved oxygen is identical between bait groups  
H₀₆: The distribution of salinity is identical between bait groups  
H₀₇: The distribution of current velocity is identical between bait groups  
H₀₈: The distribution of total length is identical between bait groups  
H₀₉: The sex ratio is identical between bait groups  
H₀₁₀: The lunar phase distribution is identical between bait groups  

The above hypotheses are tested on the full dataset and on four
sex- and size-stratified subsets (males, females, males ≥ 150cm,
females < 180cm), giving 46 tests total.

### Multiple testing correction
Bonferroni correction is applied across all 46 tests.  
Corrected significance threshold: α = 0.05 / 46 ≈ 0.001

## 2. Data description

### Full dataset description

In [5]:
df = pd.read_excel('/content/bycatch_prio_gla_ENV.xlsx', index_col=0)

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [6]:
# print info
print(f"Shape: {df.shape}")
print(f"\nDtypes:\n{df.dtypes}")

Shape: (166, 30)

Dtypes:
cod                               int64
id_bordata                       object
id_pescata                       object
barca                            object
data cattura             datetime64[ns]
Lunghezza tot (cm)              float64
Lunghezza forca (cm)            float64
INIZIO PETTORALI (cm)           float64
INIZIO I DORSALE (cm)           float64
Peso tot                        float64
Peso ev                         float64
Peso feg                        float64
Peso g                          float64
Sesso                            object
Stadio                           object
Esca                             object
Vitalità                          int64
id                                int64
y                               float64
x                               float64
depth                             int64
month                             int64
year                              int64
sst                             float64
chl           

In [7]:
print(f"\nFirst rows:")
df.head()


First rows:


,cod,id_bordata,id_pescata,barca,data cattura,Lunghezza tot (cm),Lunghezza forca (cm),INIZIO PETTORALI (cm),INIZIO I DORSALE (cm),Peso tot,...,depth,month,year,sst,chl,nppv,dox,so,curr,lunar_phase
1,1,1_7-8-19,a_7-8-19,GALEONE,2019-08-08,142.80,121.15,NaN,NaN,NaN,...,129,8,2019,26.862387,0.063426,10.918204,208.173928,38.665610,0.094294,First quarter
2,2,1_7-8-19,a_7-8-19,GALEONE,2019-08-08,158.96,130.95,NaN,NaN,NaN,...,105,8,2019,27.163318,0.078687,12.644510,207.905171,38.363757,0.127535,First quarter
3,3,1_7-8-19,a_7-8-19,GALEONE,2019-08-08,135.61,118.62,26.8,46.6,9520.0,...,124,8,2019,26.950835,0.070490,12.003047,207.782108,38.646286,0.148865,First quarter
4,4,1_7-8-19,b_7-8-19,GALEONE,2019-08-09,109.35,92.78,NaN,NaN,NaN,...,111,8,2019,27.071030,0.078635,12.810299,208.180883,38.498191,0.197662,First quarter
5,5,1_7-8-19,c_7-8-19,GALEONE,2019-08-10,121.61,99.88,23.8,51.1,NaN,...,129,8,2019,26.883981,0.064143,11.074792,208.134536,38.673369,0.091975,Waxing gibbous


In [8]:
# print missing values
missing = df.isnull().sum()
missing = missing[missing > 0]
print("Missing values per column:")
print(missing if len(missing) > 0 else "None")

Missing values per column:
Lunghezza forca (cm)      52
INIZIO PETTORALI (cm)     84
INIZIO I DORSALE (cm)     85
Peso tot                 110
Peso ev                  120
Peso feg                 122
Peso g                   124
Sesso                      8
Stadio                    48
dtype: int64


In [9]:
# class count
print("Bait counts:")
print(df['Esca'].value_counts())
print(f"\nSex counts:")
print(df['Sesso'].value_counts())

Bait counts:
Esca
Sgombro    114
Totano      52
Name: count, dtype: int64

Sex counts:
Sesso
F    87
M    71
Name: count, dtype: int64


In [35]:
print("Sex by Bait:\n")

fig, ax = plt.subplots()
df.groupby(['Sesso', 'Esca']).size().unstack().plot(
    kind='barh', stacked=True, ax=ax
)
plt.tight_layout()
plt.savefig(DRIVE_PATH + 'fig_sex_by_bait.png', dpi=150)
plt.show()

#### Sex by bait
![sex by bait](../figures/fig_sex_by_bait.png)

In [11]:
# --- Descriptive statistics per bait group -------------------
feats_cont = [
    'Lunghezza tot (cm)', 'depth', 'sst', 'chl',
    'nppv', 'dox', 'so', 'curr'
]

desc = df.groupby('Esca')[feats_cont].agg(
    ['count', 'mean', 'std', 'median',
     lambda x: x.quantile(0.25),
     lambda x: x.quantile(0.75)]
)
desc.columns.set_levels(
    ['count', 'mean', 'std', 'median', 'Q25', 'Q75'],
    level=1,
)
print(desc.round(3).T)

Esca                           Sgombro   Totano
Lunghezza tot (cm) count       114.000   52.000
                   mean        156.292  162.117
                   std          31.859   37.016
                   median      157.350  156.200
                   <lambda_0>  136.450  141.750
                   <lambda_1>  177.950  177.450
depth              count       114.000   52.000
                   mean        439.544  462.808
                   std         382.518  390.647
                   median      173.000  210.500
                   <lambda_0>  125.000  135.750
                   <lambda_1>  835.000  869.000
sst                count       114.000   52.000
                   mean         24.984   25.384
                   std           2.227    2.003
                   median       25.232   25.509
                   <lambda_0>   24.646   24.833
                   <lambda_1>   26.332   26.655
chl                count       114.000   52.000
                   mean          0.068  

In [34]:
# --- Distribution plots per feature --------------------------
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(feats_cont):
    ax = axes[i]
    for bait, color in [('Sgombro', 'steelblue'),
                        ('Totano',  'coral')]:
        vals = df.loc[df['Esca'] == bait, feat].dropna()
        ax.hist(vals, bins=15, alpha=0.5,
                label=bait, color=color, density=True)
    ax.set_title(feat)
    ax.set_xlabel('')
    ax.legend(fontsize=8)

plt.suptitle('Feature distributions by bait group — full dataset',
             y=1.02)
plt.tight_layout()
plt.savefig(DRIVE_PATH + 'fig_distributions.png', dpi=150)
plt.show()

#### Distribution plot per feature
![distribution per feature](../figures/fig_distributions.png)

In [32]:
# --- Spatial distribution of catches -------------------------
fig, ax = plt.subplots(figsize=(7, 5))
for bait, color in [('Sgombro', 'steelblue'),
                    ('Totano',  'coral')]:
    sub = df[df['Esca'] == bait]
    ax.scatter(sub['x'], sub['y'],
               label=bait, alpha=0.6, color=color, s=30)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Spatial distribution of catches by bait type')
ax.legend()
plt.tight_layout()
plt.savefig(DRIVE_PATH + 'fig_spatial.png', dpi=150)
plt.show()

#### Spatial distribution of catches
![spatial distribution](../figures/fig_spatial.png)

### Stratified subsets description

In [14]:
# drop na sex values
df.dropna(axis=0, inplace=True, subset='Sesso')

In [15]:
sex_strat = ['sex']
sex_len_strat = ['sex', 'length']

df_m, df_f = subset(df=df, divide_by=sex_strat)
df_ms, df_fs, df_mb, df_fb = subset(df=df, divide_by=sex_len_strat)

In [16]:
print(f'Male bait counts:\n {df_m.groupby('Esca').size()}')
print(f'Female bait counts:\n {df_f.groupby('Esca').size()}')

Male bait counts:
 Esca
Sgombro    48
Totano     23
dtype: int64
Female bait counts:
 Esca
Sgombro    60
Totano     27
dtype: int64


In [17]:
print(f'Male < 150 bait counts:\n {df_ms.groupby('Esca').size()}')
print(f'\nMale >= 150 bait counts: \n {df_mb.groupby('Esca').size()}')

print(f'\nFemale < 180 bait counts: \n {df_fs.groupby('Esca').size()}')
print(f'\nFemale >= 180 bait counts:\n {df_fb.groupby('Esca').size()}')

Male < 150 bait counts:
 Esca
Sgombro    13
Totano      6
dtype: int64

Male >= 150 bait counts: 
 Esca
Sgombro    35
Totano     17
dtype: int64

Female < 180 bait counts: 
 Esca
Sgombro    48
Totano     19
dtype: int64

Female >= 180 bait counts:
 Esca
Sgombro    12
Totano      8
dtype: int64


In [36]:
strat_df = [df_m, df_f, df_ms, df_fs, df_mb, df_fb]
strat_df_name = ['male', 'female', 'male < 150', 'female < 180', 'male >= 150',
                 'female >= 180']

# --- Distribution plots per feature --------------------------

markdown_lines = []

for df_strat, name in zip(strat_df, strat_df_name):

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()

    for i, feat in enumerate(feats_cont):
        ax = axes[i]
        for bait, color in [('Sgombro', 'steelblue'),
                             ('Totano',  'coral')]:
            vals = df_strat.loc[df_strat['Esca'] == bait, feat].dropna()
            ax.hist(vals, bins=15, alpha=0.5,
                    label=bait, color=color, density=True)
        ax.set_title(feat)
        ax.set_xlabel('')
        ax.legend(fontsize=8)

    plt.suptitle(f'Feature distributions by bait group — {name}', y=1.02)
    plt.tight_layout()

    # unique filename per subset
    fname = f'fig_distributions_{name.replace(" ", "_").replace(">=","gte").replace("<","lt")}.png'
    plt.savefig(DRIVE_PATH + fname, dpi=150)
    plt.show()

    # collect markdown line
    markdown_lines.append(f'#### {name}\n![{name}](../figures/{fname})')

# Print all markdown at once
print('\n\n'.join(markdown_lines))

#### male
![male](../figures/fig_distributions_male.png)

#### female
![female](../figures/fig_distributions_female.png)

#### male < 150
![male < 150](../figures/fig_distributions_male_lt_150.png)

#### female < 180
![female < 180](../figures/fig_distributions_female_lt_180.png)

#### male >= 150
![male >= 150](../figures/fig_distributions_male_gte_150.png)

#### female >= 180
![female >= 180](../figures/fig_distributions_female_gte_180.png)

In [19]:
for i, df_desc in enumerate(strat_df):
  print(f"Subset: {strat_df_name[i]}")
  print("-"*45)

  desc = df_desc.groupby('Esca')[feats_cont].agg(
      ['count', 'mean', 'std', 'median',
      lambda x: x.quantile(0.25),
      lambda x: x.quantile(0.75)]
  )
  desc.columns.set_levels(
      ['count', 'mean', 'std', 'median', 'Q25', 'Q75'],
      level=1,
  )
  print(desc.round(3).T)
  print("-"*45)

Subset: male
---------------------------------------------
Esca                           Sgombro   Totano
Lunghezza tot (cm) count        48.000   23.000
                   mean        160.934  163.007
                   std          27.981   35.215
                   median      163.050  158.960
                   <lambda_0>  148.700  148.750
                   <lambda_1>  178.725  172.750
depth              count        48.000   23.000
                   mean        503.229  508.304
                   std         390.522  396.290
                   median      272.500  504.000
                   <lambda_0>  135.250  140.000
                   <lambda_1>  885.500  867.500
sst                count        48.000   23.000
                   mean         24.738   25.119
                   std           2.204    2.231
                   median       24.994   25.038
                   <lambda_0>   24.605   24.768
                   <lambda_1>   26.203   26.658
chl                count     

## 3. Power test

Before running any test, we establish what effect sizes are detectable
given our sample sizes. This determines how to interpret non-significant
results, that is, a non-significant result in an underpowered test is
uninformative, not evidence of no effect.

We use Cohen's d as the effect size metric for continuous variables:
- 0.2 = small
- 0.5 = medium  
- 0.8 = large

For each subset we compute the minimum detectable effect size (MDE)
at 80% power and α = 0.05 (uncorrected, to give an optimistic bound).

In [20]:
subsets_n = {
    'full':          (108, 50),
    'males':         (48,  23),
    'females':       (60,  27),
    'males_gte150':  (35,  17),
    'females_lt180': (48,  19),
}

power_results = []
for subset, (n1, n2) in subsets_n.items():
    mde = tt_ind_solve_power(
        nobs1       = n1,
        ratio       = n2 / n1,
        alpha       = 0.05,
        power       = 0.8,
        alternative = 'two-sided'
    )
    power_results.append({
        'subset':    subset,
        'n_sgombro': n1,
        'n_totano':  n2,
        'MDE (Cohen d)': round(mde, 3),
        'adequately powered (d>=0.5)': mde <= 0.5
    })

pd.DataFrame(power_results)

,subset,n_sgombro,n_totano,MDE (Cohen d),adequately powered (d>=0.5)
0,full,108,50,0.482,True
1,males,48,23,0.721,False
2,females,60,27,0.657,False
3,males_gte150,35,17,0.845,False
4,females_lt180,48,19,0.771,False


## 4. Normality check

Shapiro-Wilk test applied to each continuous feature within each bait
group. This determines whether parametric (t-test) or non-parametric
(Mann-Whitney U) tests are appropriate.

Shapiro-Wilk is reliable for N < 50. Our smallest group is N = 17
(Totano, males ≥ 150cm), so the test is appropriate throughout.

H₀: the sample is drawn from a normal distribution.  
If p < 0.05 for any group → use Mann-Whitney for that feature.

In [21]:
normality_results = []
for feat in feats_cont:
  for bait in ['Sgombro', 'Totano']:
    vals = df.loc[df['Esca'] == bait, feat].dropna()
    if len(vals) >= 3:
      stat, p = shapiro(vals)
      normality_results.append({
          'feature': feat,
          'group':   bait,
          'n':       len(vals),
          'W':       round(stat, 3),
          'p':       round(p, 4),
          'normal':  p > 0.05
          })

norm_df = pd.DataFrame(normality_results)
print(norm_df.to_string(index=False))

n_non_normal = (~norm_df['normal']).sum()
print(f"\nNon-normal distributions: {n_non_normal} / {len(norm_df)}")

           feature   group   n     W      p  normal
Lunghezza tot (cm) Sgombro 108 0.974 0.0335   False
Lunghezza tot (cm)  Totano  50 0.904 0.0006   False
             depth Sgombro 108 0.771 0.0000   False
             depth  Totano  50 0.800 0.0000   False
               sst Sgombro 108 0.837 0.0000   False
               sst  Totano  50 0.847 0.0000   False
               chl Sgombro 108 0.605 0.0000   False
               chl  Totano  50 0.924 0.0033   False
              nppv Sgombro 108 0.968 0.0101   False
              nppv  Totano  50 0.964 0.1267    True
               dox Sgombro 108 0.797 0.0000   False
               dox  Totano  50 0.802 0.0000   False
                so Sgombro 108 0.963 0.0045   False
                so  Totano  50 0.732 0.0000   False
              curr Sgombro 108 0.954 0.0009   False
              curr  Totano  50 0.940 0.0136   False

Non-normal distributions: 15 / 16


### Conclusion
15 of 16 feature-group combinations deviate from normality
(Shapiro-Wilk p < 0.05) in the full dataset. Mann-Whitney U
is then used for all continuous variables in all subsequent analyses,
including stratified subsets.

## 5. Univariate tests

In [22]:
categorical_features = ['lunar_phase']
TARGET = 'Esca'
# Define subsets
subsets = {
    'full':          df,
    'males':         df_m,
    'females':       df_f,
    'males_gte150':  df_mb,
    'females_lt180': df_fs
}

# Run all tests
all_results = {}
for name, subset in subsets.items():
    all_results[name] = run_tests(
        df_subset        = subset,
        continuous_feats = feats_cont,
        categorical_feats = categorical_features,
        target           = TARGET,
        label            = name,
        bonferroni_n     = 46
    )


Subset: full  |  n=158  (108 Sgombro / 50 Totano)
Bonferroni threshold: 0.00109
           feature test  n_sgombro  n_totano  statistic      p  effect_size_r note  significant
Lunghezza tot (cm)  MWU        108        50   2451.500 0.3538          0.092             False
             depth  MWU        108        50   2510.000 0.4787          0.070             False
               sst  MWU        108        50   2292.000 0.1277          0.151             False
               chl  MWU        108        50   2619.000 0.7635          0.030             False
              nppv  MWU        108        50   2454.000 0.3587          0.091             False
               dox  MWU        108        50   3044.000 0.1991         -0.127             False
                so  MWU        108        50   2555.000 0.5891          0.054             False
              curr  MWU        108        50   2790.000 0.7379         -0.033             False
       lunar_phase Chi2        108        50      5.799

In [23]:
# --- Summary table across all subsets ------------------------
summary_rows = []
for subset_name, res_df in all_results.items():
  significant = res_df[res_df['significant'] == True]
  for _, row in significant.iterrows():
    summary_rows.append({
        'subset':    subset_name,
        'feature':   row['feature'],
        'p':         row['p'],
        'effect_r':  row.get('effect_size_r', np.nan)
    })

if summary_rows:
  print("=== Variables significant after Bonferroni correction ===")
  print(pd.DataFrame(summary_rows).to_string(index=False))
else:
  print("No variable reached significance after Bonferroni correction.")

No variable reached significance after Bonferroni correction.


## 6. Results

### Primary result
No measured variable showed a statistically significant association
with bait type after Bonferroni correction for 46 comparisons
(threshold p < 0.001), in either the full dataset or any sex- and
size-stratified subset.

### Power
The full dataset was adequately powered to detect medium effects
(MDE = 0.48, Cohen d). All stratified subsets were underpowered:
minimum detectable effect sizes ranged from 0.66 (females) to 0.85
(males ≥ 150cm), meaning only large to very large effects could have
been detected at 80% power. Null results in stratified subsets are
therefore uninformative and the absence of significance does not
constitute evidence of no effect in those populations.

### Notable patterns below the corrected threshold
Current velocity showed the most consistent directional pattern
across subsets, reaching uncorrected significance in males (p = 0.021, r = 0.342) and males ≥ 150cm (p = 0.041, r = 0.355), corresponding to small-to-medium effect sizes. Mean current velocity at capture was approximately 30% higher for Sgombro catches than Totano catches in both male subsets (males: Sgombro 0.160 ms⁻¹, Totano 0.113 ms⁻¹; males ≥150cm: Sgombro 0.154 ms⁻¹, Totano 0.102 ms⁻¹).

### Interpretation
The full dataset analysis (the only adequately powered test) returned no significant associations after correction.

On the other hand, the consistent directional pattern in current velocity across male subsets, combined with a physically substantial mean difference and
small-to-medium effect sizes, justifies exploratory follow-up despite
the failure to survive correction. A Bayesian two-group comparison
is conducted in section 7 to estimate the magnitude and credibility
of this difference without reliance on a binary significance threshold.

All remaining null results, particularly in stratified subsets, should be interpreted as inconclusive rather than negative, given insufficient statistical power to detect effects smaller than d = 0.7.

## 7. Bayesian analysis

### Rationale
The frequentist framework produced a binary outcome: no variable
survived correction. However, current velocity showed a physically
non-negligible mean difference (~30%) and a consistent directional
pattern across male subsets. The frequentist result does not
quantify how large or credible this difference is, it only
states that it does not meet an arbitrary threshold given N.

A Bayesian two-group comparison estimates the full posterior
distribution of the current difference, providing:
- the most probable magnitude of the effect and its uncertainty
- the direction of the effect

### Model
We use the BEST model (Kruschke 2013) with Student-T likelihoods, to avoid imopsing normality. The degrees-of-freedom parameter ν is estimated from the data: large ν indicates normality, small ν indicates heavy tails.

### Males

In [25]:
g0 = df_m.loc[df_m['Esca'] == 'Sgombro', 'curr']
g1 = df_m.loc[df_m['Esca'] == 'Totano', 'curr']

print(f"Sgombro: n = {len(g0)}, mean = {g0.mean():.3f}, std = {g0.std():.3f}")
print(f"Totano:  n = {len(g1)}, mean = {g1.mean():.3f}, std = {g1.std():.3f}")

with pm.Model() as best_model:

  # Priors on group means
  # Centered on pooled mean, wide enough to cover observed range
  pooled_mean = np.concatenate([g0, g1]).mean()
  pooled_std  = np.concatenate([g0, g1]).std()

  mu0 = pm.Normal('mu0', mu=pooled_mean, sigma=pooled_std*2)
  mu1 = pm.Normal('mu1', mu=pooled_mean, sigma=pooled_std*2)

  # Priors on group standard deviations
  sigma0 = pm.HalfNormal('sigma0', sigma=pooled_std)
  sigma1 = pm.HalfNormal('sigma1', sigma=pooled_std)

  # Degrees of freedom: Exponential prior favors normality
  # but allows heavy tails if data require it
  nu = pm.Exponential('nu', lam=1/30)

  # Likelihoods
  obs0 = pm.StudentT('obs0', nu=nu, mu=mu0,
                      sigma=sigma0, observed=g0)
  obs1 = pm.StudentT('obs1', nu=nu, mu=mu1,
                      sigma=sigma1, observed=g1)

  # Derived quantities
  diff = pm.Deterministic('diff', mu1 - mu0)
  cohen_d = pm.Deterministic('cohen_d',
                    diff / np.sqrt((sigma0**2 + sigma1**2) / 2))

  trace = pm.sample(2000, tune=1000,
                    target_accept=0.9,
                    return_inferencedata=True,
                    random_seed=42)

Sgombro: n = 48, mean = 0.160, std = 0.084
Totano:  n = 23, mean = 0.113, std = 0.088


Output()

In [26]:
# --- Posterior summary ---------------------------------------
print(az.summary(trace,
                 var_names=['mu0', 'mu1', 'diff', 'cohen_d', 'nu'],
                 hdi_prob=0.94))

           mean      sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  \
mu0       0.158   0.012   0.135    0.181      0.000    0.000    4278.0   
mu1       0.110   0.019   0.073    0.144      0.000    0.000    3609.0   
diff     -0.048   0.023  -0.091   -0.006      0.000    0.000    3750.0   
cohen_d  -0.583   0.282  -1.088   -0.033      0.005    0.004    3723.0   
nu       37.889  31.009   2.695   94.642      0.492    0.724    3059.0   

         ess_tail  r_hat  
mu0        2856.0    1.0  
mu1        2867.0    1.0  
diff       2689.0    1.0  
cohen_d    2868.0    1.0  
nu         2197.0    1.0  


In [37]:
# --- Posterior plots -----------------------------------------
az.plot_posterior(trace,
                  var_names=['diff', 'cohen_d', 'nu'],
                  ref_val=0,
                  hdi_prob=0.94)
plt.suptitle('Posterior distributions — current velocity, males',
             y=1.02)
plt.tight_layout()
plt.savefig(DRIVE_PATH + 'fig_bayesian_curr_males.png', dpi=150)
plt.show()

#### Posterior plots
![posterior plots male](../figures/fig_bayesian_curr_males.png)

#### Results
The posterior mean difference in current velocity (Totano − Sgombro)
was −0.048 ms⁻¹ (94% HDI: −0.091 to −0.006). The HDI excludes zero,
indicating a credible negative association: male sharks caught on
Totano were captured in locations with lower current velocity than
those caught on Sgombro (posterior means: Sgombro 0.158 ms⁻¹,
Totano 0.110 ms⁻¹). The posterior Cohen's d was −0.583 (94% HDI:
−1.088 to −0.033), corresponding to a medium effect, with the wide
credible interval reflecting uncertainty at the available sample size.
The posterior ν was 37.9 (94% HDI: 2.7 to 94.6), indicating
approximately normal distributions, with the wide HDI on ν confirming
the Student-T likelihood was the appropriate choice. Convergence
diagnostics confirmed reliable sampling (r̂ = 1.0 for all parameters,
ESS > 3000).

### Males ≥ 150

In [28]:
g0 = df_mb.loc[df_mb['Esca'] == 'Sgombro', 'curr']
g1 = df_mb.loc[df_mb['Esca'] == 'Totano', 'curr']

print(f"Sgombro: n = {len(g0)}, mean = {g0.mean():.3f}, std = {g0.std():.3f}")
print(f"Totano:  n = {len(g1)}, mean = {g1.mean():.3f}, std = {g1.std():.3f}")

with pm.Model() as best_model:

    # Priors on group means
    # Centered on pooled mean, wide enough to cover observed range
    pooled_mean = np.concatenate([g0, g1]).mean()
    pooled_std  = np.concatenate([g0, g1]).std()

    mu0 = pm.Normal('mu0', mu=pooled_mean, sigma=pooled_std*2)
    mu1 = pm.Normal('mu1', mu=pooled_mean, sigma=pooled_std*2)

    # Priors on group standard deviations
    sigma0 = pm.HalfNormal('sigma0', sigma=pooled_std)
    sigma1 = pm.HalfNormal('sigma1', sigma=pooled_std)

    # Degrees of freedom: Exponential prior favors normality
    # but allows heavy tails if data require it
    nu = pm.Exponential('nu', lam=1/30)

    # Likelihoods
    obs0 = pm.StudentT('obs0', nu=nu, mu=mu0,
                        sigma=sigma0, observed=g0)
    obs1 = pm.StudentT('obs1', nu=nu, mu=mu1,
                        sigma=sigma1, observed=g1)

    # Derived quantities
    diff = pm.Deterministic('diff', mu1 - mu0)
    cohen_d = pm.Deterministic('cohen_d',
                     diff / np.sqrt((sigma0**2 + sigma1**2) / 2))

    trace = pm.sample(2000, tune=1000,
                      target_accept=0.9,
                      return_inferencedata=True,
                      random_seed=42)

Sgombro: n = 35, mean = 0.154, std = 0.083
Totano:  n = 17, mean = 0.102, std = 0.073


Output()

In [29]:
# --- Posterior summary ---------------------------------------
print(az.summary(trace,
                 var_names=['mu0', 'mu1', 'diff', 'cohen_d', 'nu'],
                 hdi_prob=0.94))

           mean      sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  \
mu0       0.152   0.015   0.123    0.178      0.000    0.000    4256.0   
mu1       0.100   0.018   0.065    0.134      0.000    0.000    4287.0   
diff     -0.052   0.023  -0.096   -0.007      0.000    0.000    4294.0   
cohen_d  -0.684   0.315  -1.261   -0.076      0.005    0.005    4032.0   
nu       35.309  29.462   2.092   89.796      0.440    0.612    3749.0   

         ess_tail  r_hat  
mu0        2821.0    1.0  
mu1        2762.0    1.0  
diff       2909.0    1.0  
cohen_d    2944.0    1.0  
nu         2828.0    1.0  


In [38]:
# --- Posterior plots -----------------------------------------
az.plot_posterior(trace,
                  var_names=['diff', 'cohen_d', 'nu'],
                  ref_val=0,
                  hdi_prob=0.94)
plt.suptitle('Posterior distributions — current velocity, males ≥ 150cm',
             y=1.02)
plt.tight_layout()
plt.savefig(DRIVE_PATH + 'fig_bayesian_curr_big_males.png', dpi=150)
plt.show()

#### Posterior plots male
![posterior plots male](../figures/fig_bayesian_curr_big_males.png)

#### Results
The posterior mean difference in current velocity (Totano − Sgombro)
was −0.052 ms⁻¹ (94% HDI: −0.096 to −0.007). The HDI excludes zero,
indicating a credible negative association: male sharks ≥150cm caught
on Totano were captured in locations with lower current velocity than
those caught on Sgombro (posterior means: Sgombro 0.152 ms⁻¹,
Totano 0.100 ms⁻¹). The posterior Cohen's d was −0.684 (94% HDI:
−1.261 to −0.076), corresponding to a medium-to-large effect, though
the wide credible interval reflects substantial uncertainty given the
available sample size (n = 35 Sgombro, n = 17 Totano). The posterior
degrees-of-freedom parameter ν was 35.3 (94% HDI: 2.1 to 89.8),
indicating approximately normal distributions, though the wide HDI
on ν confirms the data do not strongly constrain tail behaviour,
justifying the use of the Student-T likelihood over a Normal
assumption. Convergence diagnostics confirmed reliable sampling
(r̂ = 1.0 for all parameters, ESS > 3700).

## 8. Conclusions

The current velocity association is consistent across both male
subsets: all males (n = 71, diff = −0.048 ms⁻¹, Cohen d = −0.583)
and males ≥150cm (n = 52, diff = −0.052 ms⁻¹, Cohen d = −0.684).
In both cases the 94% HDI excludes zero and the effect size is
medium to medium-large. The convergence of the pattern across two
overlapping but non-identical subsets strengthens the credibility
of the finding relative to either result in isolation.

Both analyses are exploratory. The credible intervals are wide, particularly on Cohen's d, and the smaller subset (males ≥ 150cm) drives part of the signal in the full male sample. Confirmation in an independent dataset is required
before any causal or management inference is drawn.